# Proyecto 1 – Bank Marketing

**Curso:** Inteligencia Artificial         
**Problema:** Clasificación binaria  
**Modelos:** Regresión Logística y Máquinas de Vectores de Soporte (SVM)  
**Validación:** Validación cruzada estratificada de 10 folds  
**Métricas:** Precision, Recall y F1                  
**Integrantes**: Leonardo Guevara Atehortúa y Ángel Avirama de Oro


## 1. Introducción

Este proyecto desarrolla un modelo de Machine Learning para predecir si un cliente de una institución bancaria portuguesa suscribirá un depósito a plazo después de una campaña de marketing telefónico.

El dataset Bank Marketing fue publicado por el UCI Machine Learning Repository. Contiene 45.211 registros y 17 columnas en la versión "bank-full.csv": 16 variables predictoras y una variable objetivo llamada **y**. UCI clasifica el problema como clasificación y señala que no existen valores faltantes en este dataset.

La variable objetivo es:

- **yes**: el cliente suscribió/contrató el depósito a plazo.
- **no**: el cliente no suscribió/contrató el depósito a plazo.

> **Objetivo:** construir y comparar Regresión Logística y SVM para determinar cuál ofrece el mejor desempeño según Precision, Recall y F1.


## 2. Dominio y contexto del problema

El dominio del problema es banca y marketing directo.

El banco realiza campañas telefónicas para ofrecer un producto financiero denominado "depósito a plazo". En un depósito a plazo, el cliente acepta mantener una cantidad de dinero depositada durante un periodo determinado bajo las condiciones ofrecidas por la entidad financiera.

En este contexto:

- **Cliente:** persona contactada por el banco.
- **Llamada/contacto:** interacción telefónica realizada durante una campaña de marketing.
- **Suscribir:** aceptar o contratar el depósito a plazo ofrecido.
- **Variable objetivo (y):** indica si finalmente el cliente aceptó (yes) o no (no) el producto.

El problema de Machine Learning consistirá en aprender patrones a partir de clientes históricos y utilizar esos patrones para clasificar nuevos casos.
Es decir, predecir si un cliente aceptará contratar un depósito a plazo utilizando sus características personales, financieras y la información disponible de las campañas de marketing.

### Tipo de problema

Es un problema de clasificación binaria (biclase) porque existen únicamente dos resultados posibles:

**no / yes**.


## 3. Fuente y descripción del dataset

**Fuente oficial:** UCI Machine Learning Repository : Bank Marketing.

El dataset está disponible en: https://archive.ics.uci.edu/dataset/222/bank

UCI reporta 45.211 instancias y 17 variables para **bank-full.csv**. La variable **y** es la salida que indica si el cliente suscribió un depósito a plazo.

Se trabajará con:

**bank-full.csv**

El cual contiene contiene:

- 45.211 registros.
- 16 variables predictoras.
- 1 variable objetivo.
- Tanto variables numéricas como categóricas.


La variable **duration** representa la duración de la última llamada en segundos. Es decir, esta variable está disponible después de que la llamada ocurre. Por lo tanto, como el objetivo es predecir antes de realizar la llamada qué clientes tienen mayor probabilidad de aceptar, utilizar **duration** produciría data leakage respecto al escenario de predicción.

Por esta razón, en los modelo predictivos excluiremos la variable **duration**.


In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    GridSearchCV,
    cross_validate
)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


## 5. Carga de los datos

El código siguiente carga **bank-full.csv** desde el directorio local.

In [ ]:
df = pd.read_csv("/bank-full.csv", sep=';')

print(f"Dimensiones del dataset: {df.shape[0]:,} filas x {df.shape[1]} columnas")
display(df.head())


Dimensiones del dataset: 45,211 filas x 17 columnas


,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,unknown,5,may,261,1,-1,0,unknown,no
1,44,technician,single,secondary,no,29,yes,no,unknown,5,may,151,1,-1,0,unknown,no
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,5,may,76,1,-1,0,unknown,no
3,47,blue-collar,married,unknown,no,1506,yes,no,unknown,5,may,92,1,-1,0,unknown,no
4,33,unknown,single,unknown,no,1,no,no,unknown,5,may,198,1,-1,0,unknown,no


## 6. Descripción de las variables

| Variable | Tipo | Descripción |
|---|---|---|
| **age** | Numérica entera | Edad del cliente |
| **job** | Categórica | Tipo de trabajo/ocupación |
| **marital** | Categórica | Estado civil |
| **education** | Categórica | Nivel educativo |
| **default** | Binaria/categórica | Si tiene crédito en mora |
| **balance** | Numérica entera | Saldo promedio anual en euros |
| **housing** | Binaria/categórica | Si tiene préstamo hipotecario |
| **loan** | Binaria/categórica | Si tiene préstamo personal |
| **contact** | Categórica | Tipo de contacto utilizado |
| **day** | Numérica entera | Día del mes del último contacto |
| **month** | Categórica | Mes del último contacto |
| **duration** | Numérica entera | Duración de la última llamada, en segundos |
| **campaign** | Numérica entera | Número de contactos durante la campaña actual |
| **pdays** | Numérica entera | Días desde el contacto anterior; -1 significa que no fue contactado previamente |
| **previous** | Numérica entera | Número de contactos realizados antes de la campaña actual |
| **poutcome** | Categórica | Resultado de la campaña anterior |
| **y** | **Binaria / objetivo** | Si el cliente suscribió el depósito a plazo |

**Nota:** la variable **y** es la variable de salida. Las demás son variables predictoras, aunque **duration** será excluida del modelo final.


In [ ]:
df.dtypes.to_frame("dtype")

print("\nInformación general:\n")
df.info()


Información general:

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45211 entries, 0 to 45210
Data columns (total 17 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   age        45211 non-null  int64 
 1   job        45211 non-null  object
 2   marital    45211 non-null  object
 3   education  45211 non-null  object
 4   default    45211 non-null  object
 5   balance    45211 non-null  int64 
 6   housing    45211 non-null  object
 7   loan       45211 non-null  object
 8   contact    45211 non-null  object
 9   day        45211 non-null  int64 
 10  month      45211 non-null  object
 11  duration   45211 non-null  int64 
 12  campaign   45211 non-null  int64 
 13  pdays      45211 non-null  int64 
 14  previous   45211 non-null  int64 
 15  poutcome   45211 non-null  object
 16  y          45211 non-null  object
dtypes: int64(7), object(10)
memory usage: 5.9+ MB


In [ ]:
# Valores faltantes y duplicados

missing = df.isna().sum().sort_values(ascending=False)
missing_pct = (df.isna().mean() * 100).sort_values(ascending=False)

missing_table = pd.DataFrame({
    "Valores faltantes": missing,
    "Porcentaje (%)": missing_pct.round(2)
})

print("Valores faltantes:")
display(missing_table)

print(f"Filas duplicadas: {df.duplicated().sum():,}")


Valores faltantes:


,Valores faltantes,Porcentaje (%)
age,0,0.0
job,0,0.0
marital,0,0.0
education,0,0.0
default,0,0.0
balance,0,0.0
housing,0,0.0
loan,0,0.0
contact,0,0.0
day,0,0.0


Filas duplicadas: 0
